# Milestone 2: Reproducing DANet on the NIH Malaria dataset

**Base paper:** Ali *et al.*, "DANet: a lightweight dilated attention network for malaria parasite detection",
*Scientific Reports* 15:36689 (2025). Code: <https://github.com/asfakali/DANet>

**Target:** DANet-S + LogSoftMax. Paper test-set results: ACC 97.95 %, PR 97.76 %, SE 98.07 %, SP 97.87 %, F1 97.86 %, AUC 0.98, 2.3 M params.

**Protocol** (see `BASELINE_PLAN.md` §3 and §5 for every paper-vs-code decision):
- Architecture: the authors' released `DAB_S` (cascaded dilations, channel-softmax gate, InstanceNorm). Two bugs are fixed: a `super()` NameError and a 32-way output head, now 2-way.
- Data and training: as described in the paper. Stratified 70/10/20 split (19,290 / 2,756 / 5,512), 224×224 inputs scaled to [0,1],
  rotation ±15°, H/V flips with p = 0.2, RMSprop (momentum 0) + OneCycleLR(max_lr 0.001), NLL loss, batch 64, 75 epochs, no regularisation.
  The LR follows the paper's text (0.001), not Table 2 (0.01): see §4 for the diagnostic that forced this choice.
- Evaluation: checkpoint with the lowest validation loss, evaluated **once** on the test set. **Parasitized is the positive class.**

**How to run**
- Local: run from `milestone2-baseline/`. The dataset is auto-downloaded from NIH to `data/` if missing.
- Kaggle: attach the dataset `iarunava/cell-images-for-detecting-malaria`, enable GPU and internet (internet is only needed for the patient-ID CSVs).
- Colab: the dataset auto-downloads (~350 MB).
- Smoke test: set the env var `DANET_SMOKE=1` to use 10 % of the data and 2 epochs.

In [ ]:
import os, sys, re, json, time, random, platform, urllib.request, zipfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.flop_counter import FlopCounterMode
import torchvision
from torchvision.transforms import v2
from PIL import Image
from sklearn.metrics import (confusion_matrix, f1_score, roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve)
from IPython.display import display

print(f"python {platform.python_version()} ({platform.machine()}) | torch {torch.__version__} | torchvision {torchvision.__version__}")

## 1. Configuration

In [ ]:
SMOKE = os.environ.get("DANET_SMOKE", "0") == "1"

CFG = dict(
    seed=42,
    img_size=224,
    base_channels=8,          # paper Fig. 7: 8 channels is the best trade-off
    batch_size=64,
    epochs=75,
    max_lr=0.001,             # paper text "default learning rate of 0.001"; Table 2's 0.01 does not train (see §4)
    cycle_momentum=False,     # keep RMSprop's default momentum = 0 (OneCycle would otherwise inject 0.85-0.95)
    rotation=15,
    flip_p=0.2,
    split=dict(train=0.7, val=0.1, test=0.2),
    data_frac=1.0,            # fraction of every split actually used (smoke test only)
    num_workers=0 if sys.platform == "darwin" else 2,   # macOS notebooks can't pickle notebook-defined classes
    out_dir="outputs",
)
if SMOKE:
    CFG.update(epochs=2, data_frac=0.1, out_dir="outputs_smoke")

OUT = Path(CFG["out_dir"])
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG["seed"])
print("device:", DEVICE, "| smoke test:", SMOKE)
print(json.dumps(CFG, indent=1))

## 2. Data

The NIH `cell_images` set has 27,558 cell crops from thin blood smears, 13,779 per class.
NIH also publishes patient-ID ↔ cell mappings. The paper's protocol ignores them (random cell-level split).
We load them anyway to **measure** how much patient overlap that protocol creates. This motivates M3.

In [ ]:
DATA_DIR = Path(os.environ.get("DANET_DATA", "data"))
NIH_URL = "https://data.lhncbc.nlm.nih.gov/public/Malaria/"
CLASSES = ["Parasitized", "Uninfected"]
LABEL = {"Uninfected": 0, "Parasitized": 1}      # Parasitized = positive class (label 1)

def find_class_root(base):
    """First directory under `base` whose sub-folders are exactly Parasitized/ and Uninfected/.
    This skips the nested duplicate copy that the Kaggle mirror ships."""
    for root, dirs, _ in os.walk(base):
        if sorted(d for d in dirs if not d.startswith(".")) == CLASSES:
            return Path(root)
    return None

def fetch(name, dest):
    if not dest.exists():
        print("downloading", name, "...")
        urllib.request.urlretrieve(NIH_URL + name, dest)
    return dest

IMG_ROOT = None
for cand in [DATA_DIR, Path("/kaggle/input")]:
    if cand.exists() and (IMG_ROOT := find_class_root(cand)):
        break
if IMG_ROOT is None:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    zipfile.ZipFile(fetch("cell_images.zip", DATA_DIR / "cell_images.zip")).extractall(DATA_DIR)
    IMG_ROOT = find_class_root(DATA_DIR)
print("images:", IMG_ROOT)

def load_patient_map():
    """{(class, filename): patient_id}. The NIH CSVs are 'id,[list of files],,,,' (spreadsheet export)."""
    pmap = {}
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for cls in CLASSES:
        name = f"patientid_cellmapping_{cls.lower()}.csv"
        try:
            path = fetch(name, DATA_DIR / name)
        except Exception as e:
            print(f"patient map unavailable ({e}); leakage stats will be skipped")
            return {}
        for line in open(path):
            pid = line.split(",", 1)[0].strip()
            for fn in re.findall(r"'([^']+\.png)'", line):
                pmap[(cls, fn)] = pid
    return pmap

pmap = load_patient_map()
rows = [(f"{cls}/{fn}", cls, LABEL[cls], pmap.get((cls, fn)))
        for cls in CLASSES for fn in sorted(os.listdir(IMG_ROOT / cls)) if fn.endswith(".png")]
df = pd.DataFrame(rows, columns=["rel_path", "cls", "label", "patient"])

counts = df.cls.value_counts()
print(counts.to_string())
assert (counts == 13779).all() and len(counts) == 2, "unexpected dataset contents"
if pmap:
    assert df.patient.notna().all()
    print(f"patients: {df.patient.nunique()} total | {df[df.label == 1].patient.nunique()} contribute parasitized cells")

### 2.1 Decode every image once into RAM
The images are small (~134×134 px), so all 27,558 fit in about 1.5 GB as uint8. Decoding every file also doubles as an integrity check.

In [ ]:
def read_rgb(rel):
    with Image.open(IMG_ROOT / rel) as im:
        return torch.from_numpy(np.asarray(im.convert("RGB")).copy()).permute(2, 0, 1)

t0 = time.time()
with ThreadPoolExecutor(8) as ex:
    CACHE = list(ex.map(read_rgb, df.rel_path))
sizes = np.array([img.shape[1:] for img in CACHE])
print(f"decoded {len(CACHE)} images in {time.time() - t0:.0f}s | "
      f"H×W mean {sizes.mean(0).round(0)}, min {sizes.min(0)}, max {sizes.max(0)} | "
      f"{sum(i.numel() for i in CACHE) / 1e9:.2f} GB")

### 2.2 Split: stratified 70 / 10 / 20, as in paper Table 1
Per class: 9,645 train, 1,378 val, 2,756 test. The split is saved to `splits.csv` so M3/M4 experiments use the exact same partition.

In [ ]:
def stratified_split(frame, fractions, seed):
    rng = np.random.default_rng(seed)
    split = pd.Series("train", index=frame.index)
    for _, g in frame.groupby("label"):
        idx = rng.permutation(g.index.values)
        n_test, n_val = round(fractions["test"] * len(g)), round(fractions["val"] * len(g))
        split.loc[idx[:n_test]] = "test"
        split.loc[idx[n_test:n_test + n_val]] = "val"
    return split

df["split"] = stratified_split(df, CFG["split"], CFG["seed"])
table1 = pd.crosstab(df.split, df.cls).loc[["train", "val", "test"]]
display(table1)
assert table1.Parasitized.tolist() == [9645, 1378, 2756] == table1.Uninfected.tolist(), "split differs from paper Table 1"
df.to_csv(OUT / "splits.csv", index_label="idx")

if pmap:
    train_pat = set(df[df.split == "train"].patient)
    test = df[df.split == "test"]
    print(f"Patient leakage under the paper's protocol: {test.patient.isin(train_pat).mean():.1%} of test cells "
          f"come from patients also seen in training ({len(set(test.patient) & train_pat)}/{test.patient.nunique()} test patients).")

# Subsample every split for the smoke test (full run: data_frac = 1.0, so nothing is dropped)
run_df = df if CFG["data_frac"] >= 1 else (
    df.groupby(["split", "label"], group_keys=False).sample(frac=CFG["data_frac"], random_state=CFG["seed"]))
print("samples used:", run_df.split.value_counts().to_dict())

### 2.3 Transforms and loaders
Train: resize to 224, rotate ±15°, H-flip and V-flip with p = 0.2, scale to [0,1].
Eval: resize and scale only. The repo applied the random train augmentations to val/test as well; that is fixed here.

In [ ]:
S = CFG["img_size"]
train_tf = v2.Compose([
    v2.Resize((S, S), antialias=True),
    v2.RandomRotation(CFG["rotation"]),
    v2.RandomHorizontalFlip(CFG["flip_p"]),
    v2.RandomVerticalFlip(CFG["flip_p"]),
    v2.ToDtype(torch.float32, scale=True),
])
eval_tf = v2.Compose([v2.Resize((S, S), antialias=True), v2.ToDtype(torch.float32, scale=True)])

class CellDataset(Dataset):
    def __init__(self, frame, transform):
        self.idx, self.y, self.tf = frame.index.values, frame.label.values, transform
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        return self.tf(CACHE[self.idx[i]]), int(self.y[i])

def make_loader(frame, train):
    g = torch.Generator().manual_seed(CFG["seed"])
    return DataLoader(CellDataset(frame, train_tf if train else eval_tf), batch_size=CFG["batch_size"],
                      shuffle=train, generator=g, num_workers=CFG["num_workers"],
                      pin_memory=DEVICE.type == "cuda", persistent_workers=CFG["num_workers"] > 0)

loaders = {s: make_loader(run_df[run_df.split == s], train=(s == "train")) for s in ["train", "val", "test"]}
print({s: len(l.dataset) for s, l in loaders.items()})

# Augmentation preview (cf. paper Fig. 4)
fig, axes = plt.subplots(2, 6, figsize=(12, 4.4))
for r, cls in enumerate(CLASSES):
    i = df[df.cls == cls].index[0]
    axes[r, 0].imshow(eval_tf(CACHE[i]).permute(1, 2, 0)); axes[r, 0].set_title(f"{cls}\noriginal", fontsize=9)
    for c in range(1, 6):
        axes[r, c].imshow(train_tf(CACHE[i]).permute(1, 2, 0)); axes[r, c].set_title("augmented", fontsize=9)
for a in axes.flat:
    a.axis("off")
plt.tight_layout(); plt.savefig(FIG / "augmentation.png", dpi=120); plt.show()

## 3. Model: DANet-S

This is a direct, cleaned-up port of the authors' `DAB_S` / `DA_SNet` (`model.py`). With identical weights it is
**numerically identical** to the repo model (max |Δ| = 0.0, checked during planning). Differences from the repo:
- `super(DAB_SNet, self)` NameError: fixed, by construction.
- Output layer `Linear(64, 32)` → `Linear(64, 2)`.

Inside each block: 1×1 conv, then **cascaded** 3×3 convs with dilation 1→2→3 (each followed by InstanceNorm and LeakyReLU), then concat,
then [AvgPool3 ‖ MaxPool3], then 3×3 conv, then 1×1 conv, then a **softmax over channels** producing the attention map. The map multiplies the block input, and a 1×1 projection follows.
The paper's text instead describes *parallel* dilations and a *sigmoid* gate. Both are exposed as switches for later ablations;
the baseline uses the repo's behaviour.

In [ ]:
def conv_in_act(cin, cout, k, dilation=1):
    return nn.Sequential(
        nn.Conv2d(cin, cout, k, padding=dilation * (k // 2), dilation=dilation),
        nn.InstanceNorm2d(cout, affine=True),
        nn.LeakyReLU(),
    )

class DAB_S(nn.Module):
    """Dilated Attention Block (skip variant), as implemented in the authors' repo."""
    def __init__(self, cin, cout, gate="softmax", dilation_mode="cascade"):
        super().__init__()
        assert gate in ("softmax", "sigmoid") and dilation_mode in ("cascade", "parallel")
        self.gate, self.dilation_mode = gate, dilation_mode
        self.reduce = nn.Conv2d(cin, cin, 1)                                   # f_c
        self.dilated = nn.ModuleList([conv_in_act(cin, cin, 3, d) for d in (1, 2, 3)])
        self.avg = nn.AvgPool2d(3, stride=1, padding=1)
        self.max = nn.MaxPool2d(3, stride=1, padding=1)
        self.fuse = conv_in_act(6 * cin, cin, 3)
        self.att = nn.Conv2d(cin, cin, 1)
        self.proj = conv_in_act(cin, cout, 1)

    def forward(self, x):
        h, feats = self.reduce(x), []
        for conv in self.dilated:
            src = h if (self.dilation_mode == "parallel" or not feats) else feats[-1]
            feats.append(conv(src))
        f = torch.cat(feats, 1)                                                # f_preConcat (3C)
        f = torch.cat([self.avg(f), self.max(f)], 1)                           # f_poolConcat (6C)
        a = self.att(self.fuse(f))
        a = torch.softmax(a, 1) if self.gate == "softmax" else torch.sigmoid(a)  # f_att
        return self.proj(a * x)

class DANet_S(nn.Module):
    def __init__(self, n=8, num_classes=2, **dab_kw):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, n, 3, padding=1), nn.MaxPool2d(2))
        ch = [n, n, 2*n, 2*n, 4*n, 4*n, 8*n, 8*n, 16*n, 16*n]
        layers = []
        for i in range(9):                                   # nine DABs, MaxPool after blocks 1, 3, 5, 7
            layers.append(DAB_S(ch[i], ch[i + 1], **dab_kw))
            if i in (0, 2, 4, 6):
                layers.append(nn.MaxPool2d(2))
        self.features = nn.Sequential(*layers)               # 112 → 7 spatial
        self.pool = nn.AvgPool2d(7, stride=2)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(16*n, 8*n), nn.ReLU(),
                                  nn.Linear(8*n, num_classes), nn.LogSoftmax(dim=1))

    def forward(self, x):
        return self.head(self.pool(self.features(self.stem(x))))

def count_params(m):
    return sum(p.numel() for p in m.parameters())

def count_flops(m):
    m = m.cpu().eval()
    with FlopCounterMode(display=False) as fc, torch.no_grad():
        m(torch.zeros(1, 3, S, S))
    return fc.get_total_flops()

probe = DANet_S(CFG["base_channels"])
N_PARAMS, FLOPS = count_params(probe), count_flops(probe)
print(f"params: {N_PARAMS / 1e6:.3f} M (paper: 2.3 M) | {FLOPS / 1e9:.2f} GFLOPs ({FLOPS / 2e9:.2f} GMACs) per 224² image")
assert 2.2e6 < N_PARAMS < 2.4e6
assert probe(torch.zeros(2, 3, S, S)).shape == (2, 2)

## 4. Training
RMSprop + OneCycleLR, stepped every batch. A checkpoint is written every epoch, so an interrupted run resumes where it left off.

**Why max_lr = 0.001 and `cycle_momentum=False`.** The paper is internally inconsistent: Table 2 says lr = 0.01, the text says
"default learning rate of 0.001". Also, a plain `OneCycleLR(...)` call silently cycles RMSprop's momentum between 0.85 and 0.95, while
RMSprop's own default is 0. A pre-run diagnostic (4,000 training images, 8 epochs each, same seed) gave:

| Optimiser setting | val acc after epoch 1 → 2 → 8 |
|---|---|
| RMSprop + OneCycle(max 0.01), default momentum cycling | 0.500 → 0.500 → 0.515 (stuck at chance) |
| RMSprop + OneCycle(max 0.001), default momentum cycling | 0.500 → 0.500 → 0.500 (stuck) |
| RMSprop + OneCycle(max 0.01), momentum 0 | 0.500 → 0.500 → 0.500 (stuck) |
| **RMSprop + OneCycle(max 0.001), momentum 0** | **0.736 → 0.919 → 0.964** |
| Adam 1e-3, constant (the repo's `train.py`) | 0.547 → 0.577 → 0.961 |

The stuck runs sit at loss = ln 2 with near-constant outputs. At initialisation the logit spread across images is only ≈ 0.005, and
RMSprop's un-bias-corrected first steps push more head ReLUs dead (41 % → 64 % after one step, vs 42 % with Adam). So we use the only
paper-consistent setting that trains: the text's 0.001 with RMSprop's own momentum (0).

In [ ]:
@torch.no_grad()
def predict(model, loader):
    """Returns (labels, P(parasitized), mean NLL)."""
    model.eval()
    ys, ps, loss_sum = [], [], 0.0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logp = model(x)
        loss_sum += F.nll_loss(logp, y, reduction="sum").item()
        ys.append(y.cpu()); ps.append(logp[:, 1].exp().cpu())
    y, p = torch.cat(ys).numpy(), torch.cat(ps).numpy()
    return y, p, loss_sum / len(y)

def train_model(model, train_loader, val_loader, cfg, ckpt_dir):
    ckpt_dir = Path(ckpt_dir); ckpt_dir.mkdir(parents=True, exist_ok=True)
    model.to(DEVICE)
    opt = torch.optim.RMSprop(model.parameters(), lr=cfg["max_lr"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg["max_lr"], epochs=cfg["epochs"],
                                                steps_per_epoch=len(train_loader),
                                                cycle_momentum=cfg["cycle_momentum"])
    start, hist, best = 0, [], float("inf")
    last = ckpt_dir / "last.pth"
    if last.exists():
        ck = torch.load(last, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        start, hist, best = ck["epoch"] + 1, ck["hist"], ck["best"]
        torch.set_rng_state(ck["rng"].cpu())
        print(f"resumed from epoch {start}")

    for ep in range(start, cfg["epochs"]):
        t0 = time.time()
        model.train()
        n, loss_sum, correct = 0, 0.0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logp = model(x)
            loss = F.nll_loss(logp, y)
            if not torch.isfinite(loss):
                raise RuntimeError(f"non-finite loss at epoch {ep + 1}")
            loss.backward()
            opt.step(); sched.step()
            n += len(y); loss_sum += loss.item() * len(y); correct += (logp.argmax(1) == y).sum().item()
        yv, pv, val_loss = predict(model, val_loader)
        rec = dict(epoch=ep + 1, train_loss=loss_sum / n, train_acc=correct / n, val_loss=val_loss,
                   val_acc=((pv >= 0.5) == yv).mean(), lr=sched.get_last_lr()[0], sec=time.time() - t0)
        hist.append(rec)
        if val_loss < best:
            best = val_loss
            torch.save(model.state_dict(), ckpt_dir / "best.pth")
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), epoch=ep,
                        hist=hist, best=best, rng=torch.get_rng_state()), last)
        pd.DataFrame(hist).to_csv(ckpt_dir / "train_log.csv", index=False)
        print(f"ep {ep + 1:3d}/{cfg['epochs']} | train {rec['train_loss']:.4f} / {rec['train_acc']:.4f} | "
              f"val {val_loss:.4f} / {rec['val_acc']:.4f} | lr {rec['lr']:.2e} | {rec['sec']:.0f}s"
              + ("  *best*" if best == val_loss else ""), flush=True)
    return pd.DataFrame(hist)

seed_everything(CFG["seed"])
model = DANet_S(CFG["base_channels"])
t_start = time.time()
hist = train_model(model, loaders["train"], loaders["val"], CFG, OUT / "ckpt")
print(f"training wall time (this session): {(time.time() - t_start) / 60:.1f} min")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(hist.epoch, hist.train_loss, label="train"); ax[0].plot(hist.epoch, hist.val_loss, label="val")
ax[0].set(title="NLL loss", xlabel="epoch"); ax[0].legend()
ax[1].plot(hist.epoch, hist.train_acc, label="train"); ax[1].plot(hist.epoch, hist.val_acc, label="val")
ax[1].set(title="accuracy", xlabel="epoch"); ax[1].legend()
best_ep = int(hist.loc[hist.val_loss.idxmin(), "epoch"])
for a in ax:
    a.axvline(best_ep, ls=":", c="gray")
plt.suptitle(f"DANet-S training curves (best val loss @ epoch {best_ep}; cf. paper Fig. 5)")
plt.tight_layout(); plt.savefig(FIG / "training_curves.png", dpi=120); plt.show()

## 5. Test-set evaluation (run once, best-val checkpoint)

In [ ]:
def binary_metrics(y, p, thr=0.5):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(ACC=100 * (tp + tn) / len(y), PR=100 * tp / (tp + fp), SE=100 * tp / (tp + fn),
                SP=100 * tn / (tn + fp), F1=100 * f1_score(y, pred), ROC_AUC=roc_auc_score(y, p),
                PR_AUC=average_precision_score(y, p))

model.load_state_dict(torch.load(OUT / "ckpt" / "best.pth", map_location=DEVICE))
y_test, p_test, test_loss = predict(model, loaders["test"])
ours = binary_metrics(y_test, p_test)

PAPER = dict(ACC=97.95, PR=97.76, SE=98.07, SP=97.87, F1=97.86, ROC_AUC=0.98, PR_AUC=0.98)
cmp = pd.DataFrame({"paper": PAPER, "ours": ours})
cmp["delta"] = cmp.ours - cmp.paper
display(cmp.round(4))

pred_df = run_df[run_df.split == "test"].assign(p_parasitized=p_test, pred=(p_test >= 0.5).astype(int))
pred_df.to_csv(OUT / "test_predictions.csv", index_label="idx")
json.dump(dict(cfg=CFG, metrics=ours, paper=PAPER, params=N_PARAMS, flops=FLOPS, best_epoch=best_ep,
               test_loss=test_loss), open(OUT / "results.json", "w"), indent=1, default=float)

In [ ]:
cm = confusion_matrix(y_test, (p_test >= 0.5).astype(int), labels=[1, 0])
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    ax[0].text(j, i, v, ha="center", va="center", color="white" if v > cm.max() / 2 else "black", fontsize=12)
ax[0].set(xticks=[0, 1], yticks=[0, 1], xticklabels=["Parasitized", "Uninfected"],
          yticklabels=["Parasitized", "Uninfected"], xlabel="predicted", ylabel="true", title="Confusion matrix")
fpr, tpr, _ = roc_curve(y_test, p_test)
ax[1].plot(fpr, tpr); ax[1].plot([0, 1], [0, 1], ls=":", c="gray")
ax[1].set(title=f"ROC (AUC = {ours['ROC_AUC']:.4f})", xlabel="1 − specificity", ylabel="sensitivity")
prec, rec, _ = precision_recall_curve(y_test, p_test)
ax[2].plot(rec, prec)
ax[2].set(title=f"Precision–recall (AP = {ours['PR_AUC']:.4f})", xlabel="recall", ylabel="precision")
plt.tight_layout(); plt.savefig(FIG / "test_cm_roc_pr.png", dpi=120); plt.show()

## 6. Efficiency
We have no Raspberry Pi, so single-image CPU latency (1 thread and all threads) stands in for the paper's edge-device claim (Fig. 8).

In [ ]:
def cpu_latency_ms(m, threads, reps=50):
    prev = torch.get_num_threads(); torch.set_num_threads(threads)
    m = m.cpu().eval(); x = torch.zeros(1, 3, S, S)
    with torch.no_grad():
        for _ in range(10):
            m(x)
        ts = []
        for _ in range(reps):
            t = time.perf_counter(); m(x); ts.append(time.perf_counter() - t)
    torch.set_num_threads(prev)
    return 1000 * float(np.median(ts))

eff = pd.Series({"params (M)": N_PARAMS / 1e6, "GFLOPs / image": FLOPS / 1e9,
                 "CPU latency, 1 thread (ms)": cpu_latency_ms(model, 1),
                 f"CPU latency, {torch.get_num_threads()} threads (ms)": cpu_latency_ms(model, torch.get_num_threads()),
                 "weights file (MB)": (OUT / "ckpt" / "best.pth").stat().st_size / 1e6})
display(eff.round(2).to_frame("value"))
model = model.to(DEVICE)

## 7. Grad-CAM: what the network looks at (cf. paper Fig. 9)
CAMs are computed on the last DAB (7×7 feature map) for the predicted class. The top row shows confident correct predictions;
the bottom row shows the most confident **errors**. Those errors seed the M5 failure analysis.

In [ ]:
def grad_cam(model, x, layer):
    store = {}
    def fwd_hook(_, __, out):
        store["act"] = out
        out.register_hook(lambda g: store.__setitem__("grad", g))
    h = layer.register_forward_hook(fwd_hook)
    model.eval(); model.zero_grad()
    logp = model(x)
    cls = logp.argmax(1)
    logp.gather(1, cls[:, None]).sum().backward()
    h.remove()
    w = store["grad"].mean((2, 3), keepdim=True)
    cam = F.relu((w * store["act"]).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
    cam = cam / cam.flatten(1).max(1).values.clamp_min(1e-8)[:, None, None]
    return cam.detach().cpu(), cls.cpu()

conf = np.where(pred_df.pred == 1, pred_df.p_parasitized, 1 - pred_df.p_parasitized)
pred_df = pred_df.assign(conf=conf, correct=pred_df.pred == pred_df.label)
picks = pd.concat([
    pred_df[pred_df.correct & (pred_df.label == 1)].nlargest(2, "conf"),
    pred_df[pred_df.correct & (pred_df.label == 0)].nlargest(2, "conf"),
    pred_df[~pred_df.correct].nlargest(4, "conf"),
])
x = torch.stack([eval_tf(CACHE[i]) for i in picks.index]).to(DEVICE)
cams, _ = grad_cam(model, x, model.features[-1])
name = {0: "Uninf.", 1: "Parasit."}
fig, axes = plt.subplots(2, len(picks), figsize=(2.2 * len(picks), 4.8))
for k, (i, r) in enumerate(picks.iterrows()):
    img = x[k].permute(1, 2, 0).cpu()
    axes[0, k].imshow(img); axes[0, k].set_title(f"true {name[r.label]}\npred {name[r.pred]} ({r.conf:.2f})",
                                                 fontsize=8, color="green" if r.correct else "red")
    axes[1, k].imshow(img); axes[1, k].imshow(cams[k], cmap="jet", alpha=0.45)
for a in axes.flat:
    a.axis("off")
plt.suptitle("Grad-CAM, last DAB (left 4: correct, right 4: most confident errors)")
plt.tight_layout(); plt.savefig(FIG / "gradcam.png", dpi=120); plt.show()
print(f"test errors: {(~pred_df.correct).sum()} / {len(pred_df)} "
      f"(FN = parasitized called uninfected: {((pred_df.label == 1) & ~pred_df.correct).sum()}, "
      f"FP: {((pred_df.label == 0) & ~pred_df.correct).sum()})")

## 8. Reproduction gap

Deviations from the paper that we know about, all decided before running (details in `BASELINE_PLAN.md` §3):

| Deviation | Why | Expected effect |
|---|---|---|
| Architecture follows the **released code** (cascaded dilations, channel-softmax gate, InstanceNorm), not the paper's text | The code is the artifact that matches the 2.3 M param count | Unknown. The paper never states which variant produced Table 3 |
| 2-way output layer instead of the repo's 32-way | Bug fix | ≈ none |
| Different random split (seed 42) with the same sizes as Table 1 | The paper's split was not released | ± a few tenths of a % (sampling noise on 5,512 test images is ≈ ±0.4 % at 95 % CI) |
| LR: OneCycle max_lr **0.001** (paper text) rather than Table 2's 0.01, and RMSprop momentum 0 (no OneCycle momentum cycling) | Table 2's setting and default momentum cycling both stay at chance in our diagnostic (§4) | Unknown; the authors' exact optimiser call is not public |
| PyTorch 2.x and different hardware vs. 1.9 on a T4 | Environment | ≈ none |
| Single seed | Compute budget | Run-to-run variance not captured (see the stretch section) |

### Results of the full run (seed 42, M2 MPS, 75 epochs, best val loss at epoch 47)

| Metric | Paper (single split) | Paper (5-fold CV mean, fold range) | **Ours** | Δ vs. paper split |
|---|---|---|---|---|
| Accuracy | 97.95 | 97.25 (96.97–97.42) | **97.02** (95 % CI 96.58–97.47) | −0.93 |
| Precision | 97.76 | 97.21 | **97.72** | −0.04 |
| Sensitivity | 98.07 | 97.20 | **96.30** | −1.77 |
| Specificity | 97.87 | — | **97.75** | −0.12 |
| F1 | 97.86 | 97.20 | **97.00** | −0.86 |
| ROC-AUC | 0.98 | — | **0.9945** | +0.01 |
| PR-AUC | 0.98 | — | **0.9950** | +0.01 |
| Params | 2.3 M | — | 2.31 M | ≈ 0 |

**Analysis of the observed gap**

1. **Accuracy and F1 land at the lower edge of the target band (97.0–98.5 %).** The paper's 97.95 % is outside our test-set 95 % CI (96.58–97.47 %).
   However, our 97.02 % lies **inside the paper's own 5-fold CV range (96.97–97.42 %)**. Notably, the paper's headline single-split number is higher than
   every one of its five CV folds. That suggests 97.95 % comes from a favourable split or run, and that ≈ 97.2 % is the more representative figure for this architecture.
   With one seed we cannot separate split luck from seed luck. The stretch section (CV and multiple seeds) would settle it.
2. **Almost the whole gap is sensitivity.** Precision and specificity match the paper to within 0.12 points, but we miss more parasitized cells
   (102 FN vs. 62 FP out of 5,512). This is not just a threshold effect. Even at the paper's specificity (97.87 %), our ROC curve gives SE = 96.2 %, not 98.07 %.
   Missed infections are the clinically costly error, so this is the number to watch in M3/M4.
3. **The ranking quality is actually higher than reported** (ROC-AUC 0.9945, PR-AUC 0.9950 vs. 0.98). The paper rounds to two decimals, so the
   difference may be partly cosmetic. Still, it shows the model separates the classes well; the gap is in the default operating point.
4. **The most likely sources of the gap**, roughly in order:
   (a) **Optimiser setting.** Table 2's lr 0.01 does not train at all in our hands. We used the text's 0.001 with no momentum (§4). The authors'
   exact optimiser call is not public, and the LR schedule plausibly shapes the final operating point.
   (b) **Architecture ambiguity.** We ran the released code (cascaded dilations, channel-softmax gate, InstanceNorm). The paper's text describes parallel dilations,
   a sigmoid gate and LayerNorm, and does not say which variant produced Table 3.
   (c) **Different random split and a single seed.** Our split has the same sizes as Table 1, but the paper's split was not released.
5. **Overfitting appears after about epoch 45.** Training loss falls to ≈ 0.03 while validation loss rises from 0.087 to 0.11. Best-checkpoint selection (epoch 47) protects
   the test numbers, but this contradicts the paper's statement that DANet "does not show any signs of overfitting", which was its reason for using no regularisation.
6. **Grad-CAM does not support the paper's interpretability claim.** On correctly classified parasitized cells, the last-DAB CAM is *low* on the visible
   parasite and high near the cell border (figure above). This differs from paper Fig. 9. One caveat: this is a 7×7 map taken after InstanceNorm, so CAMs from an earlier
   block may behave differently. The finding is flagged for M5 rather than treated as conclusive.
7. **Several confident errors look like label noise.** Some cells labelled uninfected show purple inclusions, and some labelled parasitized show no visible parasite.
   This matches the known NIH mislabels (cf. Fuhad et al. 2020 in the M1 survey). Errors are spread over 88 patients, and the top 5 patients account for 38 of the 164 errors.
8. **The protocol inflates every number, ours and the paper's alike.** All 201 test patients also contribute cells to the training set, so every test cell comes from a patient seen in training.
   These are therefore *within-patient* accuracies. The patient-level split is the obvious M3 hypothesis.
9. **Efficiency.** 2.31 M params and 0.83 GFLOPs confirm the "lightweight" claim. The CPU latency in §6 (≈ 92 ms with 1 thread) was measured while the
   machine was under heavy memory pressure; the smoke run measured ≈ 35 ms on the same Mac.

**Verdict:** the baseline **reproduces** at the level of the paper's own cross-validation (97.0 % vs. 97.25 % CV mean), but not its headline single-split 97.95 %.
The shortfall is concentrated in sensitivity.

## 9. Stretch experiments (off by default)
- `RUN_CV`: 5-fold CV as in paper Table 5. Each fold is the test set; 1/8 of the remainder is used for validation, keeping the 70/10/20 ratio. Costs about 5× the main run.
- `RUN_FRACTIONS`: train on 20 % of the training split with the same val/test, as in paper Table 4.

In [ ]:
RUN_CV = False
RUN_FRACTIONS = False

def run_experiment(train_df, val_df, test_df, tag, **model_kw):
    seed_everything(CFG["seed"])
    m = DANet_S(CFG["base_channels"], **model_kw)
    train_model(m, make_loader(train_df, True), make_loader(val_df, False), CFG, OUT / tag)
    m.load_state_dict(torch.load(OUT / tag / "best.pth", map_location=DEVICE))
    y, p, _ = predict(m, make_loader(test_df, False))
    return binary_metrics(y, p)

if RUN_CV:
    from sklearn.model_selection import StratifiedKFold
    folds = []
    for k, (tr, te) in enumerate(StratifiedKFold(5, shuffle=True, random_state=CFG["seed"]).split(df, df.label)):
        rest = df.iloc[tr]
        va = rest.groupby("label", group_keys=False).sample(frac=1 / 8, random_state=CFG["seed"])
        folds.append(run_experiment(rest.drop(va.index), va, df.iloc[te], f"cv_fold{k + 1}"))
    cv = pd.DataFrame(folds, index=[f"Fold {k + 1}" for k in range(5)])
    cv.loc["Average"] = cv.mean(); cv.loc["Std"] = cv.iloc[:5].std()
    display(cv.round(2)); cv.to_csv(OUT / "cv_results.csv")

if RUN_FRACTIONS:
    tr = df[df.split == "train"].groupby("label", group_keys=False).sample(frac=0.2, random_state=CFG["seed"])
    frac20 = run_experiment(tr, df[df.split == "val"], df[df.split == "test"], "frac20")
    display(pd.DataFrame({"20% train": frac20, "paper 20%": dict(ACC=97.06, SE=96.87, SP=97.24, PR=97.09, F1=96.98)}).round(2))